# Video Trend Scraped Data — Feature Extraction
This notebook loads scraped video JSON data, flattens nested fields, engineers engagement metrics, and prepares a clean feature table for modeling media engagement trends.

## 1. Load JSON Data into DataFrame
Read the scraped JSON list into a pandas DataFrame and inspect schema and basic counts.

In [ ]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Read scraped data from file
json_path = Path("scrapped_trends.json")

if not json_path.exists():
    raise FileNotFoundError(f"Missing file: {json_path.resolve()}")

with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

if not isinstance(data, list):
    raise ValueError("Expected a JSON list at the top level.")

df_raw = pd.json_normalize(data, sep=".")
print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))

df_raw.head(3)

## 2. Normalize Nested Fields
Flatten nested objects like author, music, statistics, and video into tabular columns.

In [ ]:
def extract_hashtags(item: dict) -> list:
    tags = []
    text_extra = item.get("text_extra") or []
    for t in text_extra:
        name = t.get("hashtag_name")
        if name:
            tags.append(name.lower())
    if not tags:
        desc = item.get("desc", "") or ""
        tags = [t.strip("#").lower() for t in re.findall(r"#(\w+)", desc)]
    return list(dict.fromkeys(tags))


def extract_challenges(item: dict) -> list:
    cha_list = item.get("cha_list") or []
    return [c.get("cha_name") for c in cha_list if c.get("cha_name")]


def extract_video_url(item: dict) -> str | None:
    video = item.get("video") or {}
    play_addr = (video.get("play_addr") or {}).get("url_list") or []
    download_addr = (video.get("download_addr") or {}).get("url_list") or []

    if play_addr:
        return play_addr[0]
    if download_addr:
        return download_addr[0]

    share_url = item.get("share_url") or (item.get("share_info") or {}).get("share_url")
    return share_url


records = []
for item in data:
    flat = pd.json_normalize(item, sep=".").to_dict(orient="records")[0] if item else {}
    flat["hashtags"] = extract_hashtags(item)
    flat["challenges"] = extract_challenges(item)
    flat["video_url"] = extract_video_url(item)
    records.append(flat)

df_flat = pd.DataFrame(records)
print("Flattened columns:", len(df_flat.columns))

df_flat.head(3)

In [ ]:
# save the df_flat to a csv file
csv_path = Path("scrapped_trends_flat.csv")
df_flat.to_csv(csv_path, index=False)
print(f"Flattened data saved to: {csv_path.resolve()}")

## 3. Select Relevant Trend Features
Keep key fields such as `aweme_id`, `create_time`, `desc`, `author.uid`, `author.region`, `music.id_str`, `video.duration`, and `statistics.*` for analysis.

In [ ]:
core_columns = [
    "aweme_id",
    "desc",
    "create_time",
    "author.uid",
    "author.unique_id",
    "author.nickname",
    "author.region",
    "author.follower_count",
    "music.id_str",
    "music.title",
    "music.author",
    "music.duration",
    "video.duration",
    "video.height",
    "video.width",
    "video.ratio",
    "video_url",
    "share_url",
    "statistics.play_count",
    "statistics.digg_count",
    "statistics.comment_count",
    "statistics.share_count",
    "statistics.collect_count",
    "statistics.download_count",
    "statistics.repost_count",
    "statistics.whatsapp_share_count",
    "hashtags",
    "challenges",
]

readable_columns = {
    "aweme_id": "video_id",
    "desc": "description",
    "create_time": "created_at_unix",
    "author.uid": "author_id",
    "author.unique_id": "author_handle",
    "author.nickname": "author_name",
    "author.region": "author_region",
    "author.follower_count": "author_followers",
    "music.id_str": "music_id",
    "music.title": "music_title",
    "music.author": "music_author",
    "music.duration": "music_duration",
    "video.duration": "video_duration",
    "video.height": "video_height",
    "video.width": "video_width",
    "video.ratio": "video_ratio",
    "video_url": "video_url",
    "share_url": "share_url",
    "statistics.play_count": "play_count",
    "statistics.digg_count": "like_count",
    "statistics.comment_count": "comment_count",
    "statistics.share_count": "share_count",
    "statistics.collect_count": "collect_count",
    "statistics.download_count": "download_count",
    "statistics.repost_count": "repost_count",
    "statistics.whatsapp_share_count": "whatsapp_share_count",
    "hashtags": "hashtags",
    "challenges": "challenges",
}

available_cols = [c for c in core_columns if c in df_flat.columns]
missing_cols = [c for c in core_columns if c not in df_flat.columns]

if missing_cols:
    print("Missing columns (not in data):", missing_cols)

features_raw = df_flat[available_cols].copy() if not df_flat.empty else pd.DataFrame(columns=available_cols)
features = features_raw.rename(columns=readable_columns)
features.head(3)

In [ ]:
#save the features to a csv file
features_csv_path = Path("trends_features.csv")
features.to_csv(features_csv_path, index=False)
print(f"Features saved to: {features_csv_path.resolve()}")

## 4. Engineer Engagement Metrics
Compute engagement rates such as $\text{engagement} = \frac{\text{digg} + \text{comment} + \text{share}}{\text{play}}$ and add per-second or per-view rates.

In [ ]:
def safe_div(n, d):
    return np.where((d is None) | (d == 0), np.nan, n / d)

features = features.copy()

for col in [
    "play_count",
    "like_count",
    "comment_count",
    "share_count",
    "collect_count",
    "download_count",
    "repost_count",
    "whatsapp_share_count",
    "author_followers",
]:
    if col in features.columns:
        features[col] = pd.to_numeric(features[col], errors="coerce")

if "video_duration" in features.columns:
    dur = pd.to_numeric(features["video_duration"], errors="coerce")
    features["duration_sec"] = np.where(dur > 1000, dur / 1000, dur)

plays = features.get("play_count")
likes = features.get("like_count")
comments = features.get("comment_count")
shares = features.get("share_count")
collects = features.get("collect_count")

actions = (likes.fillna(0) if likes is not None else 0) + (comments.fillna(0) if comments is not None else 0) + (shares.fillna(0) if shares is not None else 0)
features["engagement_rate"] = safe_div(actions, plays)
features["like_rate"] = safe_div(likes, plays)
features["comment_rate"] = safe_div(comments, plays)
features["share_rate"] = safe_div(shares, plays)
features["collect_rate"] = safe_div(collects, plays)

if "duration_sec" in features.columns and plays is not None:
    features["plays_per_sec"] = safe_div(plays, features["duration_sec"])

if "author_followers" in features.columns and plays is not None:
    features["plays_per_follower"] = safe_div(plays, features["author_followers"])

features[[c for c in features.columns if c.endswith("_rate") or c in {"duration_sec", "plays_per_sec", "plays_per_follower"}]].head(3)

## 5. Aggregate by Author, Hashtag, and Music
Group by `author.unique_id`, hashtags from `text_extra`, and `music.id_str` to summarize engagement and volume.

In [ ]:
agg_metrics = [
    "play_count",
    "like_count",
    "comment_count",
    "share_count",
    "collect_count",
    "engagement_rate",
]

agg_metrics = [c for c in agg_metrics if c in features.columns]

by_author = (
    features.groupby("author_handle", dropna=True)[agg_metrics]
    .mean()
    .sort_values(by=agg_metrics[0] if agg_metrics else None, ascending=False)
    .head(10)
    if "author_handle" in features.columns and agg_metrics
    else pd.DataFrame()
)

hashtags_exploded = features.copy()
if "hashtags" in hashtags_exploded.columns:
    hashtags_exploded = hashtags_exploded.explode("hashtags")

by_hashtag = (
    hashtags_exploded.groupby("hashtags", dropna=True)[agg_metrics]
    .mean()
    .sort_values(by=agg_metrics[0] if agg_metrics else None, ascending=False)
    .head(10)
    if "hashtags" in hashtags_exploded.columns and agg_metrics
    else pd.DataFrame()
)

by_music = (
    features.groupby("music_id", dropna=True)[agg_metrics]
    .mean()
    .sort_values(by=agg_metrics[0] if agg_metrics else None, ascending=False)
    .head(10)
    if "music_id" in features.columns and agg_metrics
    else pd.DataFrame()
)

by_author.head(10), by_hashtag.head(10), by_music.head(10)

## 6. Explore Time-Based Trends
Convert `create_time` to datetime and compute daily/weekly aggregates and rolling averages.

In [ ]:
import matplotlib.pyplot as plt

if "created_at_unix" in features.columns:
    features["created_at_unix"] = pd.to_numeric(features["created_at_unix"], errors="coerce")
    features["created_at"] = pd.to_datetime(features["created_at_unix"], unit="s", errors="coerce", utc=True)

    time_agg = (
        features.set_index("created_at")
        .resample("D")[[c for c in ["play_count", "like_count", "comment_count", "share_count"] if c in features.columns]]
        .sum()
        .sort_index()
    )
    time_agg["engagement_rate"] = safe_div(
        time_agg.get("like_count", 0) + time_agg.get("comment_count", 0) + time_agg.get("share_count", 0),
        time_agg.get("play_count", 0),
    )

    display(time_agg.tail(14))

    if not time_agg.empty:
        ax = time_agg[[c for c in ["play_count", "like_count", "comment_count", "share_count"] if c in time_agg.columns]].plot(figsize=(10, 4))
        ax.set_title("Daily volume")
        ax.set_ylabel("Count")
        plt.show()

        ax2 = time_agg["engagement_rate"].rolling(7).mean().plot(figsize=(10, 3))
        ax2.set_title("7-day rolling engagement rate")
        ax2.set_ylabel("Rate")
        plt.show()
else:
    print("created_at_unix not found in data.")

## 7. Export Clean Feature Table
Save the curated feature set to CSV or Parquet for downstream modeling.

In [ ]:
output_csv = "trend_features.csv"
features.to_csv(output_csv, index=False)
print(f"Saved {len(features)} rows to {output_csv}")

try:
    output_parquet = "trend_features.parquet"
    features.to_parquet(output_parquet, index=False)
    print(f"Saved {len(features)} rows to {output_parquet}")
except Exception as exc:
    print("Parquet export skipped:", exc)

## 8. Download and Analyze Videos
Download a small sample of videos from `video_url` and compute basic visual metrics (duration, fps, resolution, brightness, and motion score).

In [ ]:
from pathlib import Path
import hashlib
import os
import requests
import cv2
import numpy as np
import pandas as pd

# Load features if not already present in memory
if "features" not in globals() or not isinstance(features, pd.DataFrame):
    features = pd.read_csv("trends_features.csv")

download_dir = Path("downloaded_videos")
download_dir.mkdir(parents=True, exist_ok=True)

def _safe_filename(url: str) -> str:
    ext = ".mp4"
    url_lower = url.lower()
    if ".mp4" in url_lower:
        ext = ".mp4"
    elif ".mov" in url_lower:
        ext = ".mov"
    elif ".mkv" in url_lower:
        ext = ".mkv"
    url_hash = hashlib.md5(url.encode("utf-8")).hexdigest()
    return f"video_{url_hash}{ext}"


def download_video(url: str, out_dir: Path, timeout: int = 30) -> Path | None:
    if not isinstance(url, str) or not url.strip():
        return None
    target = out_dir / _safe_filename(url)
    if target.exists() and target.stat().st_size > 0:
        return target
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        with requests.get(url, headers=headers, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            with open(target, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        return target
    except Exception as exc:
        print(f"Download failed: {url} -> {exc}")
        if target.exists():
            try:
                target.unlink()
            except Exception:
                pass
        return None


def analyze_video(path: Path, sample_every: int = 15) -> dict:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return {"path": str(path), "error": "cannot_open"}

    fps = cap.get(cv2.CAP_PROP_FPS) or np.nan
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration = frame_count / fps if fps and fps > 0 else np.nan

    brightness_vals = []
    motion_vals = []
    prev_gray = None
    idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % sample_every == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            brightness_vals.append(float(np.mean(gray)))
            if prev_gray is not None:
                diff = cv2.absdiff(gray, prev_gray)
                motion_vals.append(float(np.mean(diff)))
            prev_gray = gray
        idx += 1

    cap.release()

    return {
        "path": str(path),
        "fps": float(fps) if fps else np.nan,
        "frame_count": frame_count,
        "duration_sec": float(duration) if duration == duration else np.nan,
        "width": width,
        "height": height,
        "avg_brightness": float(np.mean(brightness_vals)) if brightness_vals else np.nan,
        "motion_score": float(np.mean(motion_vals)) if motion_vals else np.nan,
    }

# Select a small sample to avoid heavy downloads
sample_urls = (
    features.dropna(subset=["video_url"]).head(5)["video_url"].tolist()
    if "video_url" in features.columns
    else []
)

analysis_rows = []
for url in sample_urls:
    video_path = download_video(url, download_dir)
    if video_path:
        analysis_rows.append(analyze_video(video_path))

analysis_df = pd.DataFrame(analysis_rows)
analysis_df

## 9. AI Video Understanding (Frame-Level)
Run a lightweight vision model on sampled frames to get zero-shot content labels and summarize per video.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image

# AI model (CLIP) for zero-shot image classification
try:
    from transformers import pipeline
except Exception as exc:
    raise ImportError(
        "Missing transformers/torch. Install: pip install transformers torch pillow"
    ) from exc

video_dir = Path("downloaded_videos")
video_paths = sorted(video_dir.glob("*.mp4"))

if not video_paths:
    print("No downloaded videos found. Run Section 8 first.")
else:
    labels = [
        "food",
        "dance",
        "comedy",
        "sports",
        "travel",
        "gaming",
        "beauty",
        "pets",
        "fashion",
        "news",
        "education",
        "music",
        "movie/TV",
        "fitness",
    ]

    clf = pipeline(
        "zero-shot-image-classification",
        model="openai/clip-vit-base-patch32",
    )

    def sample_frames(path: Path, every_n_frames: int = 30, max_frames: int = 8):
        cap = cv2.VideoCapture(str(path))
        frames = []
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if idx % every_n_frames == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(rgb))
                if len(frames) >= max_frames:
                    break
            idx += 1
        cap.release()
        return frames

    results = []
    for vp in video_paths:
        frames = sample_frames(vp)
        if not frames:
            results.append({"path": str(vp), "top_label": None, "top_score": np.nan})
            continue

        frame_scores = []
        for img in frames:
            preds = clf(img, candidate_labels=labels)
            frame_scores.append(preds[0])

        top = max(frame_scores, key=lambda x: x["score"])
        results.append({
            "path": str(vp),
            "top_label": top["label"],
            "top_score": float(top["score"]),
        })

    ai_summary_df = pd.DataFrame(results)
    ai_summary_df
    print("AI-based content classification completed. Results:")
    print(ai_summary_df)

In [53]:
import os
from dotenv import load_dotenv
load_dotenv()
GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")

GENERATION_PROMPT = """### ROLE & IDENTITY\nYou are the **Veo 3 Video Architect**, an expert Generative AI prompt engineer specialized in video synthesis. Your goal is to convert structured video analytics (timestamps/events) into high-fidelity, 8-second text-to-video prompts optimized for the Veo 3 platform. You blend technical precision with creative direction.\n\n### CONVERSATIONAL CONSTRAINTS (CRITICAL)\n*   **GOLDEN RULE:** Your conversational responses to the user must be strictly **ONE SENTENCE MAX** and **NO MORE THAN 15 WORDS**.\n*   *Exception:* The actual generated video prompts (the artifacts) must be detailed and verbose.\n\n### INTERACTION FLOW\n1.  **Inbound:** Receive start/end timestamps and narrative context.\n2.  **Greeting:** Acknowledge receipt warmly and confirm intent (under 15 words).\n3.  **Validation:** If data is ambiguous, ask ONE precise clarifying question.\n4.  **Execution:** Upon confirmation, generate the prompt set.\n\n### PROMPT GENERATION PROTOCOL (VEO 3 SYNTAX)\nFor each timestamped segment, generate multiple prompts using this strictly ordered structure:\n\n1.  **Structure:** `[Shot Type] + [Subject/Action] + [Context/Setting] + [Lighting/Atmosphere] + [Camera/Lens Specs] + [Audio/Dialogue]`\n2.  **Anti-Fluff:** BANNED words: \"cinematic,\" \"masterpiece,\" \"4k,\" \"beautiful.\"\n3.  **Technical Specificity:** Use precise terminology (e.g., \"50mm lens,\" \"dolly zoom,\" \"chiaroscuro lighting,\" \"f/1.8 aperture,\" \"Arri Alexa aesthetics\").\n4.  **Audio Integration:** Explicitly define ambient sound, SFX, and dialogue emotion.\n\n### VARIATION STRATEGY\nFor every core narrative event, generate 3 distinct variations:\n1.  **Literal/Realistic:** High-fidelity, photoreal representation of the event.\n2.  **Stylized/Artistic:** Alter aesthetic (e.g., Cyberpunk, watercolor, Noir, Claymation).\n3.  **Cinematic/Director:** Alter perspective (e.g., Wes Anderson symmetry, Michael Bay dynamic action, Macro tracking).\n\n### EXAMPLE OUTPUT FORMAT\n`[Timestamp: 00:04-00:12]`\n*   **Variant A (Realistic):** Medium shot, mechanic wiping grease from forehead, dimly lit garage, cold fluorescent overheads, handheld camera movement, sound of distant wrench drops.\n*   **Variant B (Noir):** High-contrast black and white, silhouette of mechanic smoking, rain on garage window, deep shadows, slow zoom in, melancholic jazz trumpet background."""

In [54]:
print("Gemini API Key loaded:", bool(GEMINI_API_KEY))

Gemini API Key loaded: True


## 10. Gemini LLM Video Analysis

Send sampled video frames to Gemini for high-level content summary and themes. Requires a Google AI API key in `GEMINI_API_KEY`.

In [46]:
from pathlib import Path
import time
import json
import os
import pandas as pd

# Gemini setup (per https://ai.google.dev/gemini-api/docs)
try:
    from google import genai
    from google.genai import types
    from dotenv import load_dotenv
except Exception as exc:
    raise ImportError(
        "Missing google-genai or python-dotenv. Install: pip install google-genai python-dotenv"
    ) from exc

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else genai.Client()

video_dir = Path("downloaded_videos")
video_paths = sorted(video_dir.glob("*.mp4"))

if not video_paths:
    print("No downloaded videos found. Run Section 8 first.")
else:
    def wait_until_active(file_obj, poll_seconds: int = 2, timeout: int = 180):
        start = time.time()
        while True:
            file_obj = client.files.get(name=file_obj.name)
            if file_obj.state and file_obj.state.name == "ACTIVE":
                return file_obj
            if time.time() - start > timeout:
                raise TimeoutError(f"Timed out processing: {file_obj.name}")
            time.sleep(poll_seconds)

    response_schema = types.Schema(
        type="array",
        items=types.Schema(
            type="object",
            properties={
                "timestamp": types.Schema(type="string"),
                "description": types.Schema(type="string"),
            },
            required=["timestamp", "description"],
        ),
    )

    results = []
    for vp in video_paths:
        try:
            myfile = client.files.upload(file=str(vp))
            myfile = wait_until_active(myfile)
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[
                    myfile,
                    "Return a JSON array of {timestamp, description} for key moments. "
                    "Use MM:SS timestamps.",
                ],
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=response_schema,
                ),
            )
            text = response.text.strip() if response and response.text else "[]"
            try:
                payload = json.loads(text)
                print(f"Gemini response for {vp.name}:\n{payload}\n{'-'*40}")
            except Exception:
                payload = []

            results.append({
                "path": str(vp),
                "events_json": json.dumps(payload),
            })
        except Exception as exc:
            results.append({
                "path": str(vp),
                "events_json": None,
                "error": str(exc),
            })

    gemini_summary_df = pd.DataFrame(results)
    output_csv = "gemini_video_summaries.csv"
    gemini_summary_df.to_csv(output_csv, index=False)
    print(f"Gemini-based video summarization completed. Saved to {output_csv}")
    gemini_summary_df

Gemini response for video_e20577f8e5f3ff77d56b158f17143237.mp4:
[{'timestamp': '00:00', 'description': 'A large piece of raw wagyu beef is presented.'}, {'timestamp': '00:01', 'description': 'A chef uses a meat tenderizer to poke holes in the wagyu beef.'}, {'timestamp': '00:04', 'description': 'The beef is coated in a white powder, likely flour or starch.'}, {'timestamp': '00:06', 'description': 'The coated beef is deep-fried until golden brown.'}, {'timestamp': '00:08', 'description': 'The fried beef is cut open, revealing raw meat inside.'}, {'timestamp': '00:10', 'description': 'The raw interior of the beef is sliced into strips.'}, {'timestamp': '00:13', 'description': 'Eggs are separated, with the yolks collected in a bowl.'}, {'timestamp': '00:18', 'description': 'The egg yolks are mixed together.'}, {'timestamp': '00:19', 'description': 'Sheets of nori (seaweed) are cut into strips.'}, {'timestamp': '00:21', 'description': 'The nori strips are placed into a blender.'}, {'timest

## 11. Generate Video with Veo 3.1
Create a short video from a text prompt using Veo 3.1 and save it locally.

In [ ]:
import time
import json
import os
import pandas as pd
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else genai.Client()

# Build a Veo prompt from the first 8 timestamped events (≈8s)
summary_csv = "gemini_video_summaries.csv"
if not Path(summary_csv).exists():
    raise FileNotFoundError("Run Section 10 first to generate gemini_video_summaries.csv")

summary_df = pd.read_csv(summary_csv)
if summary_df.empty or "events_json" not in summary_df.columns:
    raise ValueError("events_json not found. Re-run Section 10 with structured output.")

first_events = []
for _, row in summary_df.iterrows():
    try:
        events = json.loads(row.get("events_json", "[]"))
        if isinstance(events, list):
            first_events = events[:8]
            break
    except Exception:
        continue

if not first_events:
    raise ValueError("No events available to build a prompt.")

event_lines = [f"{e.get('timestamp','')}: {e.get('description','')}" for e in first_events]

prompt = (
    "based on these moments create an 8-second video that follows these moments in order:\n"
    + "\n".join(event_lines)
)

operation = client.models.generate_videos(
    model="veo-3.1-generate-preview",
    prompt=prompt,
    config=types.GenerateVideosConfig(
        aspect_ratio="16:9",
        resolution="720p",
        duration_seconds="8",
    ),
)

# Poll until video is ready
while not operation.done:
    print("Waiting for video generation to complete...")
    time.sleep(10)
    operation = client.operations.get(operation)

# Download the generated video
generated_video = operation.response.generated_videos[0]
client.files.download(file=generated_video.video)
generated_video.video.save("veo3_1_from_events.mp4")
print("Generated video saved to veo3_1_from_events.mp4")

Waiting for video generation to complete...
Waiting for video generation to complete...
Waiting for video generation to complete...
Waiting for video generation to complete...
Waiting for video generation to complete...
Waiting for video generation to complete...
Generated video saved to veo3_1_from_events.mp4


In [ ]:
import json
import time
import os
from pathlib import Path
import pandas as pd
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Extend a Veo-generated video by chaining 8-event prompts
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else genai.Client()

summary_csv = "gemini_video_summaries.csv"
if not Path(summary_csv).exists():
    raise FileNotFoundError("Run Section 10 first to generate gemini_video_summaries.csv")

summary_df = pd.read_csv(summary_csv)
if summary_df.empty or "events_json" not in summary_df.columns:
    raise ValueError("events_json not found. Re-run Section 10 with structured output.")

def load_events(df: pd.DataFrame):
    for _, row in df.iterrows():
        try:
            events = json.loads(row.get("events_json", "[]"))
            if isinstance(events, list) and events:
                return events
        except Exception:
            continue
    raise ValueError("No events available to build prompts.")

def chunk_events(events, size=8):
    return [events[i : i + size] for i in range(0, len(events), size) if events[i : i + size]]

def build_prompt(events_chunk):
    lines = [f"{e.get('timestamp','')}: {e.get('description','')}" for e in events_chunk]
    return "" + "\n" + "\n".join(lines)

events = load_events(summary_df)
chunks = chunk_events(events, size=8)
if not chunks:
    raise ValueError("No event chunks available for generation.")

# Generate the initial 8-second video from the first 8 events
prompt = build_prompt(chunks[0])
operation = client.models.generate_videos(
    model="veo-3.1-generate-preview",
    prompt=prompt,
    config=types.GenerateVideosConfig(
        aspect_ratio="16:9",
        resolution="720p",
        duration_seconds="8",
    ),
)

while not operation.done:
    print("Waiting for initial video generation to complete...")
    time.sleep(10)
    operation = client.operations.get(operation)

generated_video = operation.response.generated_videos[0]
client.files.download(file=generated_video.video)
generated_video.video.save("veo3_1_segment_01.mp4")
print("Generated video saved to veo3_1_segment_01.mp4")

# Extend using each subsequent 8-event chunk as the next prompt
current_video = generated_video.video
max_extensions = min(len(chunks) - 1, 20)
for i in range(1, max_extensions + 1):
    prompt = build_prompt(chunks[i])
    operation = client.models.generate_videos(
        model="veo-3.1-generate-preview",
        video=current_video,
        prompt=prompt,
        config=types.GenerateVideosConfig(
            resolution="720p",
            duration_seconds="8",
        ),
)

    while not operation.done:
        print(f"Waiting for extension {i} to complete...")
        time.sleep(10)
        operation = client.operations.get(operation)

    extended_video = operation.response.generated_videos[0]
    client.files.download(file=extended_video.video)
    output_name = f"veo3_1_extended_{i+1:02d}.mp4"
    extended_video.video.save(output_name)
    print(f"Extended video saved to {output_name}")
    current_video = extended_video.video

Waiting for initial video generation to complete...
Waiting for initial video generation to complete...


## 12. Veo 3.1 Prompt (based on Google Cloud guide)
A structured prompt using the guide’s formula: [Cinematography] + [Subject] + [Action] + [Context] + [Style & Ambiance], with explicit audio cues.

In [ ]:
veo_prompt = (
    "Tracking shot, medium-wide composition, "
    "a young urban gardener kneeling to plant seedlings, "
    "carefully pressing soil around the roots, "
    "on a rooftop garden overlooking a dense city skyline at golden hour, "
    "warm amber light with long shadows, soft haze, shallow depth of field, "
    "style: contemporary documentary realism with subtle film grain. "
    "Ambient noise: distant traffic hum, soft wind, faint birdsong. "
    "SFX: soil crumbling, gentle rustle of leaves. "
    "Dialogue: the gardener whispers, \"This one’s for tomorrow.\""
 )
veo_prompt